# Decode is memory-bound
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week4/Decode_Batch_Sweep_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 4, Module 2: Prefill, decode and ridge points**

Each decode step multiplies every weight matrix by one vector per request. Each weight is read from memory once per step and used for one multiply-add per request, so at batch size 1 decode does about one FLOP per byte. That is far below the ridge point of any GPU, so the step time is set by how fast the weights can be read. Batching shares that read across requests: the bytes stay almost the same, and the work grows with the batch. Each step also reads every request's KV cache, which grows with context and is not shared.

This notebook does the measure, model and compare steps from the Lab 4 slide. You predict the step time of Qwen 2.5 0.5B from its bytes and the T4's measured bandwidth, then measure it at several batch sizes and context lengths, and explain the gap.

**What you will do**
1. Measure the T4's memory bandwidth with a copy.
2. Compute the bytes one decode step reads: weights plus KV cache.
3. Measure decode step time from batch size 1 to 64.
4. Measure it from 128 to 2048 tokens of context.
5. Compare the measurements with the model, and separate GPU time from time outside the GPU.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The notebook takes about three minutes.

## 1. Check the environment and load the model

In [ ]:
import math
import statistics
import time
import torch
from torch.profiler import profile, ProfilerActivity
from transformers import AutoModelForCausalLM

assert torch.cuda.is_available(), "This notebook needs a GPU: Runtime > Change runtime type > T4 GPU"
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).cuda().eval()
cfg = model.config
print(f"{cfg.num_hidden_layers} layers, {cfg.num_attention_heads} query heads, "
      f"{cfg.num_key_value_heads} KV heads, hidden size {cfg.hidden_size}")

## 2. Measured bandwidth
The roofline uses peak numbers, but real kernels run below them. The cell copies a 256 MB tensor and counts both the read and the write. The T4's published bandwidth is 320 GB/s.

In [ ]:
src = torch.empty(256 * 2**20, dtype=torch.uint8, device="cuda")
dst = torch.empty_like(src)
for _ in range(3):
    dst.copy_(src)
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(20):
    dst.copy_(src)
torch.cuda.synchronize()
bw = 2 * src.numel() * 20 / (time.perf_counter() - t0)
del src, dst
print(f"Measured bandwidth: {bw / 1e9:.0f} GB/s ({bw / 320e9:.0%} of the published 320 GB/s)")

## 3. The bytes one decode step reads
Two terms, from the lecture:

- **Weights.** Every parameter is read once per step, whatever the batch size. At FP16 that is 2 bytes per parameter.
- **KV cache.** Each request reads its own keys and values for every token of context: `2 x layers x KV heads x head dim x 2 bytes` per token.

Qwen 2.5 uses grouped-query attention with 2 KV heads for 14 query heads, so its KV cache is 7 times smaller than it would be with multi-head attention. The cell prints both.

In [ ]:
weight_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
head_dim = getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads
kv_per_token = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * head_dim * 2
kv_per_token_mha = 2 * cfg.num_hidden_layers * cfg.num_attention_heads * head_dim * 2
n_params = weight_bytes / 2

print(f"Weights read per step:   {weight_bytes / 1e6:7.0f} MB")
print(f"KV cache per token:      {kv_per_token:7,d} bytes  (with {cfg.num_key_value_heads} KV heads)")
print(f"Same model with MHA:     {kv_per_token_mha:7,d} bytes  ({cfg.num_attention_heads} KV heads)")


def predict(batch, ctx):
    """Lower bound on step time, and arithmetic intensity, from bytes and measured bandwidth."""
    nbytes = weight_bytes + batch * ctx * kv_per_token
    flops = 2 * n_params * batch + 4 * cfg.num_hidden_layers * cfg.hidden_size * ctx * batch
    return nbytes / bw * 1e3, flops / nbytes


for batch in (1, 8, 64):
    ms, ai = predict(batch, 256)
    print(f"batch {batch:2d}, 256 tokens of context: at least {ms:.2f} ms per step, {ai:5.1f} FLOPs per byte")
print("T4 ridge point (65 TFLOPS / 320 GB/s): about 203 FLOPs per byte")

**Reading the output**
- The weights are about 1 GB. At the measured bandwidth, reading them takes a few milliseconds, which is the lower bound on a decode step.
- Arithmetic intensity is close to the batch size: about 1 FLOP per byte at batch 1 and about 60 at batch 64. All of these are below the T4's ridge point, so decode stays memory-bound.

## 4. Measuring a decode step
The helper fills a KV cache with a prefill pass over a random prompt, then times decode steps. It reports two numbers:

- **wall time:** the time per step measured with the CPU clock, with the GPU synchronized. This is what a user would see.
- **GPU time:** the time the GPU spent running kernels in one step, from the profiler.

The prefill calls the transformer body directly, without the output layer, so it does not build logits for every prompt position. Only the decode steps compute logits, for one token per request.

In [ ]:
def gpu_ms(prof):
    total = 0.0
    for e in prof.key_averages():
        if e.device_type == torch.autograd.DeviceType.CPU:
            us = getattr(e, "self_device_time_total", None)
            total += (us if us is not None else e.self_cuda_time_total) / 1e3
    return total


def measure(batch, ctx, steps=8):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    ids = torch.randint(0, cfg.vocab_size, (batch, ctx), device="cuda")
    with torch.inference_mode():
        out = model.model(input_ids=ids, use_cache=True)      # prefill: fill the KV cache
        state = {"cache": out.past_key_values,
                 "tok": model.lm_head(out.last_hidden_state[:, -1:]).argmax(-1)}

        def step():
            o = model(input_ids=state["tok"], past_key_values=state["cache"], use_cache=True)
            state["cache"] = o.past_key_values
            state["tok"] = o.logits[:, -1:].argmax(-1)

        for _ in range(3):
            step()                                             # warm-up
        times = []
        for _ in range(steps):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            step()
            torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1e3)
        with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
            for _ in range(3):
                step()
            torch.cuda.synchronize()
    result = {"wall": statistics.median(times), "gpu": gpu_ms(prof) / 3,
              "mem": torch.cuda.max_memory_allocated() / 1e9}
    del state, out
    return result


measure(1, 128)   # warm-up the whole path once
print("ready")

## 5. Sweep the batch size
Context is fixed at 256 tokens. If decode is memory-bound, the step time should barely change as the batch grows, because the weights dominate the bytes and are read once for the whole batch.

In [ ]:
batch_rows = []
print(f"{'batch':>5} {'predicted':>10} {'GPU time':>9} {'wall time':>10} {'tokens/s':>9} {'memory':>7}")
for batch in (1, 2, 4, 8, 16, 32, 64):
    r = measure(batch, 256)
    pred, _ = predict(batch, 256)
    batch_rows.append((batch, pred, r["gpu"], r["wall"]))
    print(f"{batch:5d} {pred:7.2f} ms {r['gpu']:6.2f} ms {r['wall']:7.2f} ms {batch / r['wall'] * 1e3:9.0f} {r['mem']:5.1f} GB")

**Reading the output**
- The predicted time grows by only a few percent from batch 1 to 64. The weights are shared, and the KV cache of this model is small.
- The GPU time is about twice the prediction and roughly flat up to batch 16. Real kernels do not reach the copy bandwidth, and attention, layer norms and the output layer add work the model leaves out.
- At batch 32 and 64 the GPU time starts to rise. The work that grows with the batch, such as the output layer over the 151,936-word vocabulary and the activations, is no longer small next to the weight read.
- Tokens per second grows almost in proportion to the batch. This is why serving systems batch decode.
- The wall time is about three times the GPU time, and it does not change with the batch. Between kernels, the GPU waits for Python and the PyTorch dispatcher to issue the next one. Section 7 comes back to this.

## 6. Sweep the context length
Batch is fixed at 8. Each request now reads more KV cache per step.

In [ ]:
ctx_rows = []
print(f"{'context':>7} {'KV read':>8} {'predicted':>10} {'GPU time':>9} {'wall time':>10}")
for ctx in (128, 512, 1024, 2048):
    r = measure(8, ctx)
    pred, _ = predict(8, ctx)
    ctx_rows.append((ctx, pred, r["gpu"], r["wall"]))
    print(f"{ctx:7d} {8 * ctx * kv_per_token / 1e6:5.0f} MB {pred:7.2f} ms {r['gpu']:6.2f} ms {r['wall']:7.2f} ms")
mha_ms = (weight_bytes + 8 * 2048 * kv_per_token_mha) / bw * 1e3
print(f"\nWith multi-head attention, batch 8 at 2048 tokens would read "
      f"{8 * 2048 * kv_per_token_mha / 1e6:.0f} MB of KV per step: at least {mha_ms:.2f} ms.")

**Reading the output**
- At batch 8 and 2048 tokens, the KV cache read is about 200 MB, a fifth of the weights. The step time rises with context, but slowly, because grouped-query attention keeps the cache small.
- The last line shows the same step with one KV head per query head: the cache read would be 7 times larger, about 1.4 GB, more than the weights. The lecture's 13B example uses multi-head attention, which is why its KV cache passes its weights at about 4,000 tokens.

## 7. Model versus measurement
The plot puts the prediction, the GPU time and the wall time on one chart for the batch sweep.

In [ ]:
import matplotlib.pyplot as plt

b = [r[0] for r in batch_rows]
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(b, [r[1] for r in batch_rows], "k--", label="predicted (bytes / measured bandwidth)")
ax.plot(b, [r[2] for r in batch_rows], "o-", color="0.35", label="GPU time per step")
ax.plot(b, [r[3] for r in batch_rows], "s-", color="0.7", label="wall time per step")
ax.set_xscale("log", base=2)
ax.set_xlabel("batch size (256 tokens of context)")
ax.set_ylabel("ms per decode step")
ax.set_ylim(bottom=0)
ax.legend(frameon=False, fontsize=8)
ax.set_title("Decode step time barely depends on batch size")
plt.tight_layout()
plt.show()

gap = statistics.median(r[2] / r[1] for r in batch_rows)
host = statistics.median(r[3] / r[2] for r in batch_rows)
print(f"GPU time is about {gap:.1f}x the bandwidth bound. Wall time is about {host:.1f}x the GPU time.")

**Reading the output**
- **Measured versus modeled GPU time.** The GPU time is about twice the bound. Part of the gap is that kernels reach less than the copy bandwidth. Part is work the model leaves out: attention, layer norms, the output layer over a 151,936-word vocabulary, and small kernels that are limited by latency rather than bandwidth.
- **Wall time versus GPU time.** This gap is not a memory effect. It is time outside the GPU, the overhead term that Week 6 analyzes. A compiled decode step with CUDA Graphs closes most of it.
- Keep the measured and modeled values separate when you report them, as the Lab 4 slide asks.

## Try this
1. Load the model in FP32 (`dtype=torch.float32`) and run section 5 again. The weights double, and so should the GPU time.
2. In section 6, try batch 32 at 2048 tokens and check the memory column. How close are you to the T4's 15 GB?
3. Compute the batch size at which this model's decode would reach the T4 ridge point, from `predict`. Is it reachable within the T4's memory?

## Summary
- A decode step reads every weight once. At batch 1, arithmetic intensity is about 1 FLOP per byte, far below the ridge.
- Batching shares the weight read, so step time stays nearly flat and throughput grows with the batch.
- Each request's KV cache is read every step and is not shared. Grouped-query attention makes it small for this model.
- The bandwidth bound predicts the GPU time to within a small factor. The rest of the wall time is overhead outside the GPU.

Lecture: Week 4, Module 1, slides 8 to 11 (ridge point, decode intensity with and without KV), Module 2, slides 15 and 16 (prefill versus decode, ridge points compared), and the Lab 4 slide.